<a href="https://colab.research.google.com/github/Barusik/testing/blob/main/Google%20meet%20info.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New section

In [1]:
!pip install google-api-python-client google-auth google-auth-httplib2 google-auth-oauthlib

In [6]:
from google.colab import files
uploaded = files.upload()  # vybereš JSON na počítači
list(uploaded.keys())

Saving deep-dispatch-468409-f2-5862588aadb2.json to deep-dispatch-468409-f2-5862588aadb2.json


['deep-dispatch-468409-f2-5862588aadb2.json']

In [11]:
SERVICE_ACCOUNT_FILE = "/content/deep-dispatch-468409-f2-5862588aadb2.json"   # pokud se jmenuje jinak, uprav
DELEGATED_USER = "info@filozofieprojektu.cz"          # ten, za koho se „vydáváme“
SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]

In [17]:
# --- NASTAVENÍ ---
SERVICE_ACCOUNT_FILE = "/content/deep-dispatch-468409-f2-5862588aadb2.json"   # uprav, pokud se jmenuje jinak
DELEGATED_USER = "info@filozofieprojektu.cz"          # účet z vaší domény
CAL_ID = "primary"                                   # nebo konkrétní ID kalendáře

from google.oauth2 import service_account
from googleapiclient.discovery import build
from datetime import datetime, timedelta, timezone
import json

# Časové okno: posledních 30 dní až dalších 60 dní
time_min = (datetime.now(timezone.utc) - timedelta(days=30)).isoformat()
time_max = (datetime.now(timezone.utc) + timedelta(days=60)).isoformat()

# Přihlášení + impersonace
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=["https://www.googleapis.com/auth/calendar.readonly"]
).with_subject(DELEGATED_USER)
svc = build("calendar", "v3", credentials=creds)

# Volitelně si můžeš vypsat dostupné kalendáře a vzít si jejich ID:
# cal_list = svc.calendarList().list().execute()
# print(json.dumps([{ "summary": c.get("summary"), "id": c.get("id")} for c in cal_list.get("items", [])], indent=2, ensure_ascii=False))

# Načtení událostí včetně e-mailů hostů
resp = svc.events().list(
    calendarId=CAL_ID,
    timeMin=time_min,
    timeMax=time_max,
    singleEvents=True,
    orderBy="startTime",
    maxResults=100,
    maxAttendees=100,         # důležité pro plné seznamy hostů
    alwaysIncludeEmail=True   # snaž se vždy vrátit email
).execute()

items = resp.get("items", [])
print(f"Nalezeno {len(items)} událostí v {CAL_ID} mezi {time_min} a {time_max}.\n")

for ev in items:
    start = ev["start"].get("dateTime", ev["start"].get("date"))
    title = ev.get("summary") or "(bez názvu)"
    attendees = ev.get("attendees", [])
    emails = [a.get("email") for a in attendees if a.get("email")]
    print(f"- {start}  {title}")
    print(f"  Účastníci: {', '.join(emails) if emails else '— (žádní/neviditelní)'}")
    # Pokud budeš chtít debug jednoho eventu:
    # print(json.dumps(ev, indent=2, ensure_ascii=False)[:1200])

Nalezeno 2 událostí v primary mezi 2025-07-09T10:49:45.369363+00:00 a 2025-10-07T10:49:45.369425+00:00.

- 2025-08-04T11:30:00+02:00  Meet meeting
  Účastníci: barbora.vackova85@gmail.com, info@filozofieprojektu.cz
- 2025-08-12T13:00:00+02:00  Schůzka test
  Účastníci: barbora.vackova@omnihouse.eu, info@filozofieprojektu.cz
